In [23]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt
from langgraph.prebuilt import ToolNode, tools_condition
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langgraph.constants import START, END
from langgraph.graph import StateGraph, MessagesState
from dotenv import load_dotenv

load_dotenv()


# ====== Step 1: 定义各种工具 ======
@tool
def check_inbox() -> str:
    """Read an email from the given address."""
    # 模拟收件箱邮件
    return str([
        {
            "subject": "周末见个面？",
            "content": """
                嗨 虎哥，
                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？

                祝好，简
            """,
            "from": "jane@itcast.cn",
            "status": "unread"
        },
        {
            "subject": "周五会议",
            "content": """
                嗨 虎哥，
                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？

                祝好，小李
            """,
            "from": "lixiaolong@itcast.cn",
            "status": "checked"
        }
    ])


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    human_decision = interrupt({
        "args": {
            "to": to,
            "subject": subject,
            "body": body,
        },
        "confirm": "这是我帮你写的邮件，你觉得如何？",
        "actions": ["reject", "approve", "edit"]
    })
    decision = human_decision.get("action")
    if decision == "approve":
        return f"邮件已发送至 {to} , 主题： {subject} , 内容： {body}"
    elif decision == "reject":
        return f"用户拒绝发送：{human_decision.get("reason")}"
    else:  # 修改邮件了，需要读取邮件内容
        body = human_decision['args']['body']
        return f"邮件已发送至 {to} , 主题： {subject} , 内容： {body}"


# 组成工具数组
tools = [check_inbox, send_email]

# ====== Step 2: 定义模型，绑定工具 ======
# 定义模型
model = init_chat_model("deepseek-chat")
# 绑定工具到模型
model_with_tools = model.bind_tools(tools)


# ====== Step 3: 定义Node ======
def llm_node(state: MessagesState):
    """LLM节点：调用模型，可以选择调用工具"""
    response = model_with_tools.invoke(state["messages"])
    return {"messages": [response]}


# ====== Step 4: 构建图 ======
builder = StateGraph(MessagesState)
builder.add_node("llm", llm_node)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", tools_condition)
builder.add_edge("tools", "llm")  # 工具执行后回到LLM，形成循环
hitl_graph = builder.compile(checkpointer=InMemorySaver())

In [24]:
# ====== Step 4: 测试 ======
config = {"configurable": {"thread_id": "1"}}
response = hitl_graph.invoke(
    {"messages": [HumanMessage("帮我查看下邮件")]},
    config=config,
)

print(response['messages'][-1].text)

你的收件箱里有 2 封邮件：

**1. 未读 — 来自 jane@itcast.cn**
- 主题：周末见个面？
- 内容：Jane 说她下周会去城里，问有没有机会一起喝杯咖啡。

**2. 已读 — 来自 lixiaolong@itcast.cn**
- 主题：周五会议
- 内容：小李为无法准时参加周五的会议道歉，询问能否重新安排一个时间。

需要我帮你回复其中哪一封吗？


In [25]:
response = hitl_graph.invoke(
    {"messages": [HumanMessage("帮我回复Jane，告诉她很高兴她能来，问问具体时间")]},
    config=config,
)

print(response['messages'][-1].text)

In [26]:
interrupt_info = response['__interrupt__'][0].value
print(interrupt_info)

{'args': {'to': 'jane@itcast.cn', 'subject': 'Re: 周末见个面？', 'body': '嗨 简，\n\n很高兴收到你的消息，也很高兴你要来城里！一起喝杯咖啡当然没问题。\n\n方便告诉我你具体的行程安排吗？比如哪天、什么时间段你比较有空，我好提前安排一下。\n\n期待见面！\n\n祝好，虎哥'}, 'confirm': '这是我帮你写的邮件，你觉得如何？', 'actions': ['reject', 'approve', 'edit']}


In [27]:
from langgraph.types import Command

# 拒绝
response = hitl_graph.invoke(
    Command(resume={"action": "reject", "reason": "太正式了！Jane是我的女友"}),
    config=config,
)
print(response['messages'][-1].text)
if response.get('__interrupt__'):
    interrupt_info = response['__interrupt__'][0].value
    print(interrupt_info)

原来如此，那我换个轻松点的口吻：
{'args': {'to': 'jane@itcast.cn', 'subject': 'Re: 周末见个面？', 'body': '简，\n\n太好了，你能来我超开心！咖啡必须安排上 ☕\n\n你具体啥时候到、哪几天有空？说个时间我好提前腾出来。\n\n想你了，早点定下来～\n\n虎哥'}, 'confirm': '这是我帮你写的邮件，你觉得如何？', 'actions': ['reject', 'approve', 'edit']}


In [28]:
response = hitl_graph.invoke(
    Command(resume={"action": "approve"}),
    config=config,
)

In [29]:
print(response['messages'][-1].text)
if response.get('__interrupt__'):
    interrupt_info = response['__interrupt__'][0].value
    print(interrupt_info)

已经发给 Jane 了，语气轻松亲切一些：

> 简，
>
> 太好了，你能来我超开心！咖啡必须安排上 ☕
>
> 你具体啥时候到、哪几天有空？说个时间我好提前腾出来。
>
> 想你了，早点定下来～
>
> 虎哥

如果觉得哪里还想调整（比如更简洁或者更俏皮），跟我说，我再改一版发过去。另外小李那封关于周五会议改期的邮件还没回，需要处理的话告诉我。
